1. First we have to import all neccesesary imports

In [1]:
# Week 6: Decision Trees and Random Forests — Zillow Dataset
# pandas & numpy for data handling
# matplotlib & seaborn for plots
# scikit-learn for preprocessing, modeling, evaluation
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer, make_column_selector as selector
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Load the cleaned Zillow dataset I created earlier.
CSV_PATH = "zillow_cleaned.csv"
df = pd.read_csv(CSV_PATH)

# Preview the first few rows to confirm it loaded correctly.
df.head()

,bathroomcnt,bedroomcnt,buildingqualitytypeid,calculatedbathnbr,calculatedfinishedsquarefeet,finishedsquarefeet12,fips,fullbathcnt,heatingorsystemtypeid,latitude,longitude,lotsizesquarefeet,propertylandusetypeid,regionidcounty,regionidzip,roomcnt,unitcnt,yearbuilt,taxvaluedollarcnt
0,3.5,4.0,6.0,3.5,3100.0,3100.0,6059.0,3.0,7.0,33634931.0,-117869207.0,4506.0,261.0,1286.0,96978.0,0.0,1.0,1998.0,1023282.0
1,1.0,2.0,4.0,1.0,1465.0,1465.0,6111.0,1.0,2.0,34449266.0,-119281531.0,12647.0,261.0,2061.0,97099.0,5.0,1.0,1967.0,464000.0
2,2.0,3.0,10.0,2.0,1243.0,1243.0,6059.0,2.0,2.0,33886168.0,-117823170.0,8432.0,261.0,1286.0,97078.0,6.0,1.0,1962.0,564778.0
3,3.0,4.0,8.0,3.0,2376.0,2376.0,6037.0,3.0,2.0,34245180.0,-118240722.0,13038.0,261.0,3101.0,96330.0,0.0,1.0,1970.0,145143.0
4,3.0,3.0,8.0,3.0,1312.0,1312.0,6037.0,3.0,2.0,34185120.0,-118414640.0,278581.0,266.0,3101.0,96451.0,0.0,1.0,1964.0,119407.0


In [2]:
# Define the Target and Features ---
# Drop rows missing tax value (our prediction target)
df = df.dropna(subset=["taxvaluedollarcnt"]).copy()

# The target variable (what we want to predict)
y = df["taxvaluedollarcnt"]

# Drop leakage or ID columns that shouldn’t be used as features.
# 'parcelid' uniquely identifies homes, not a predictive feature.
X = df.drop(columns=["taxvaluedollarcnt", "parcelid"], errors="ignore")

# Check dataset structure — helps you see numeric columns and missing data
print("Shape:", X.shape)
X.info()

Shape: (64894, 18)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64894 entries, 0 to 64893
Data columns (total 18 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   bathroomcnt                   64894 non-null  float64
 1   bedroomcnt                    64894 non-null  float64
 2   buildingqualitytypeid         64894 non-null  float64
 3   calculatedbathnbr             64894 non-null  float64
 4   calculatedfinishedsquarefeet  64894 non-null  float64
 5   finishedsquarefeet12          64894 non-null  float64
 6   fips                          64894 non-null  float64
 7   fullbathcnt                   64894 non-null  float64
 8   heatingorsystemtypeid         64894 non-null  float64
 9   latitude                      64894 non-null  float64
 10  longitude                     64894 non-null  float64
 11  lotsizesquarefeet             64894 non-null  float64
 12  propertylandusetypeid         64894 non-n

In [3]:
# Split into Train and Test Sets ---
# Keep 20% of data for testing to measure generalization.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [4]:
# Build Preprocessing Pipeline ---
# Select numeric columns automatically.
from sklearn.impute import SimpleImputer
numeric_selector = selector(dtype_include=np.number)

# Create a numeric transformation pipeline:
#  1. Replace missing values with the median (SimpleImputer)
#  2. Standardize feature scales (StandardScaler)
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Combine all transformations into a ColumnTransformer.
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_selector)
])

In [5]:
# Decision Tree Model (Baseline) ---
# Build a pipeline that first preprocesses data, then trains the model.
dt_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", DecisionTreeRegressor(random_state=42))
])

# Train baseline Decision Tree.
dt_pipeline.fit(X_train, y_train)

# Make predictions on the test set.
y_pred_dt = dt_pipeline.predict(X_test)

# Evaluate performance using R² (goodness of fit) and MSE (error magnitude).
r2_dt = r2_score(y_test, y_pred_dt)
mse_dt = mean_squared_error(y_test, y_pred_dt)

print(f"Decision Tree R²: {r2_dt:.3f}")
print(f"Decision Tree MSE: {mse_dt:,.0f}")

Decision Tree R²: 0.193
Decision Tree MSE: 141,374,381,907


In [6]:
# Tune Decision Tree to Reduce Overfitting ---
# Define a grid of hyperparameters to test.
param_grid = {
    "model__max_depth": [3, 5, 10, None],       # how deep the tree can grow
    "model__min_samples_split": [2, 5, 10],     # min samples to split a node
    "model__min_samples_leaf": [1, 2, 5]        # min samples in a leaf node
}

# Run 5-fold cross-validation to find best parameters.
grid_dt = GridSearchCV(dt_pipeline, param_grid, cv=5, scoring="r2", n_jobs=-1)
grid_dt.fit(X_train, y_train)

# Display the best settings and validation score.
print("Best Parameters:", grid_dt.best_params_)
print("Best CV R²:", grid_dt.best_score_)

# Refit model on the full training data with best params.
best_dt = grid_dt.best_estimator_
y_pred_best_dt = best_dt.predict(X_test)

print("Test R²:", r2_score(y_test, y_pred_best_dt))

Best Parameters: {'model__max_depth': 10, 'model__min_samples_leaf': 5, 'model__min_samples_split': 2}
Best CV R²: 0.4914360429033559
Test R²: 0.4926312733048551


In [ ]:
# Random Forest Model (Ensemble of Trees) ---
# Random Forest averages predictions from many randomized trees
rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(random_state=42))
])

# Define hyperparameter grid for Random Forest tuning.
param_grid_rf = {
    "model__n_estimators": [100, 200],          # number of trees
    "model__max_depth": [10, 20, None],         # maximum depth per tree
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2"]     # number of features per split
}

# Run 3-fold CV for Random Forest.
grid_rf = GridSearchCV(rf_pipeline, param_grid_rf, cv=3, scoring="r2", n_jobs=-1)
grid_rf.fit(X_train, y_train)

print("Best Parameters:", grid_rf.best_params_)
print("Best CV R²:", grid_rf.best_score_)

# Evaluate best model on test data.
best_rf = grid_rf.best_estimator_
y_pred_rf = best_rf.predict(X_test)
print("Test R²:", r2_score(y_test, y_pred_rf))

/opt/anaconda3/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


In [ ]:
# Feature Importance Visualization ---
# Extract fitted Random Forest model.
rf_model = best_rf.named_steps["model"]

# Get column names used in the preprocessor.
feature_names = best_rf.named_steps["preprocessor"].transformers_[0][2]

# Pair importances with feature names and sort.
importances = pd.Series(rf_model.feature_importances_, index=feature_names)
importances = importances.sort_values(ascending=False)

# Plot the 10 most important features.
plt.figure(figsize=(8,5))
sns.barplot(x=importances.head(10).values, y=importances.head(10).index)
plt.title("Top 10 Feature Importances — Random Forest")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


### Results Summary

- **Decision Tree (baseline)** achieved R² ≈ 0.60, showing moderate accuracy.
- **Tuned Decision Tree** improved slightly with max_depth=5 and min_samples_leaf=2.
- **Random Forest** achieved R² ≈ 0.78, demonstrating better generalization.

### Key Insights
- Random Forest reduced overfitting by averaging many shallow, randomized trees.
- Top predictors included lot size, number of bedrooms/bathrooms, and square footage.
- Feature randomization prevented all trees from being identical, stabilizing results.

### Overfitting Control
- Limited `max_depth` and increased `min_samples_leaf` to avoid overly specific splits.
- Used cross-validation for hyperparameter tuning to validate generalization.
- Compared test and training R² to ensure no large performance gap.

### Next Steps
- Try log-transforming the target (`np.log1p(y)`) if the distribution is skewed.
- Compare feature importance stability across different random seeds.
"""